# Milvus GPU Test
Testing Milvus GPU index creation, data insertion, and searching before implementing it in the main codebase.

In [1]:
import random
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility

# 1. Connect to Milvus
connections.connect(alias="default", host="localhost", port="19530")
print("Connected to Milvus.")

collection_name = "test_gpu_collection"

# 2. Clean up old testing collection
if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)
    print(f"Dropped existing collection {collection_name}")

# 3. Define schema
# Assuming embedding dimension might be 2048 as an example. Adjust based on your model's actual dim.
DIMENSION = 2048

id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True)
text_field = FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=1000)
embedding_field = FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=DIMENSION)

schema = CollectionSchema(fields=[id_field, text_field, embedding_field], description="Test GPU index collection")

# 4. Create collection
collection = Collection(name=collection_name, schema=schema)
print(f"Created collection {collection_name}.")

# 5. Create GPU Index
# Using a GPU-accelerated index. Milvus GPU versions support indices like GPU_IVF_FLAT, GPU_IVF_PQ, GPU_CAGRA.
index_params = {
    "metric_type": "L2",       # Or IP (Inner Product), COSINE
    "index_type": "GPU_IVF_FLAT",  # Changed from standard IVF_FLAT to GPU_IVF_FLAT
    "params": {"nlist": 1024}
}

collection.create_index(field_name="embedding", index_params=index_params)
print("GPU index created on the embedding field.")

Connected to Milvus.
Created collection test_gpu_collection.
GPU index created on the embedding field.


In [2]:
import numpy as np
import time

# Let's generate a batch of random dummy embeddings (using numpy, shape: total_docs x dim)
total_docs = 50000
print(f"Generating {total_docs} dummy embeddings of size {DIMENSION} to stress test insertion and indexing...")

# We normalize random uniformly distributed values or standard normals
embeddings_batch = np.random.rand(total_docs, DIMENSION).astype(np.float32).tolist()
texts_batch = [f"This is a sample document text number {i}" for i in range(total_docs)]

# 6. Insert data in chunks to avoid gRPC size limits (64MB)
start_time = time.time()
chunk_size = 5000
for i in range(0, total_docs, chunk_size):
    end = min(i + chunk_size, total_docs)
    collection.insert([texts_batch[i:end], embeddings_batch[i:end]])
    print(f"Inserted chunk {i} to {end}...")
    
print(f"Total insertion completed in {time.time() - start_time:.2f} seconds.")

# 7. Flush data and load into memory for search
start_time = time.time()
collection.flush()
collection.load()
print(f"Collection flushed and loaded in {time.time() - start_time:.2f} seconds.")

# 8. Performing GPU vector search
query_vector = np.random.rand(1, DIMENSION).astype(np.float32).tolist()

search_params = {"metric_type": "L2", "params": {"nprobe": 10}}
start_time = time.time()
results = collection.search(
    data=query_vector, 
    anns_field="embedding", 
    param=search_params, 
    limit=5, 
    output_fields=["text"]
)
search_time = time.time() - start_time

print(f"\nSearch complete in {search_time:.4f} seconds (limit=5).")
for hits in results:
    for hit in hits:
        print(f" - Dist: {hit.distance:.4f} | Text: '{hit.entity.get('text')}'")
        
# Example complete! You can run this and monitor metrics to ensure GPU is being utilized.
# Remember to clean it up!
# collection.drop()

Generating 50000 dummy embeddings of size 2048 to stress test insertion and indexing...
Inserted chunk 0 to 5000...
Inserted chunk 5000 to 10000...
Inserted chunk 10000 to 15000...
Inserted chunk 15000 to 20000...
Inserted chunk 20000 to 25000...
Inserted chunk 25000 to 30000...
Inserted chunk 30000 to 35000...
Inserted chunk 35000 to 40000...
Inserted chunk 40000 to 45000...
Inserted chunk 45000 to 50000...
Total insertion completed in 8.29 seconds.
Collection flushed and loaded in 18.01 seconds.

Search complete in 0.0216 seconds (limit=5).
 - Dist: 308.1415 | Text: 'This is a sample document text number 5520'
 - Dist: 308.4160 | Text: 'This is a sample document text number 10458'
 - Dist: 309.3310 | Text: 'This is a sample document text number 23168'
 - Dist: 309.7020 | Text: 'This is a sample document text number 24197'
 - Dist: 309.7336 | Text: 'This is a sample document text number 21504'


In [3]:
collection.drop()